## Project Modules

In [1]:
from project_modules.project_imports import *
from pathlib import Path

from project_modules.classes import Patient
from project_modules.classes import Clinic

from project_modules.rules import call_a_rule

from project_modules.simulation_tools import create_appointments, plot_line_graph
from project_modules.simulation_tools import establish_attendance, random_patient_sample
from project_modules.simulation_tools import get_margin_errors, check_convergence_mean, confidence_interval, calculate_summary

## Prediction Model

In [2]:
X_file_path = "prediction_model/X.npy"
y_file_path = "prediction_model/y.npy"
column_file_path = "prediction_model/column_names.pkl"

with open(column_file_path, 'rb') as file:
    column_names = pickle.load(file)

X = np.load(X_file_path, allow_pickle=True)
y = np.load(y_file_path, allow_pickle=True)

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

X = pd.DataFrame(X)
X.columns = column_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777)

model_file_path = "prediction_model/Final_model.pkl"
with open(model_file_path, 'rb') as file:
    model = pickle.load(file)

model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)
y_pred_proba_1 = y_pred_proba[:,-1]

assert roc_auc_score(y_test, y_pred_proba_1) > 0

## Initialization

In [3]:
patients_original = [Patient(i, **row) for i, row in enumerate(X.to_dict(orient='records'))]
patients = patients_original.copy()

# Extracting keys from the first patient object
patient_keys = set(patients[0].__dict__.keys())

# Converting column_names to a set for faster comparison
column_name_set = set(column_names)

# Finding keys that are in patient_keys but not in column_names
missing_keys = patient_keys - column_name_set

verbose = False
if verbose:
    print("Keys in patients[0].__dict__.keys() but not in column_names:")
    print(missing_keys)
    len(missing_keys)

In [4]:
'''
# Si se necesita volver a calcular las probabilidades (~1min 39s)
patients_with_proba = []
for patient in patients:
    patient.predict_proba(model)
    patients_with_proba.append(patient)

# export patients_with_proba to a pkl file
import pickle
with open('patients_with_proba_may5.pkl', 'wb') as f:
    pickle.dump(patients_with_proba, f)
''' 

# Cargando con las probabilidades ya hechas
with open('patients_with_proba.pkl', 'rb') as f:
    patients_with_proba = pickle.load(f)

patients = patients_with_proba.copy()
probas = [patient.proba for patient in patients]

plotting = False
if plotting:
    plt.figure(figsize=(10,5))
    plt.hist(probas, bins=50)
    plt.xlabel('Predicted probability')
    plt.ylabel('Number of patients')
    plt.title('Predicted probability distribution')
    plt.show()

ppv_npv_results_file_path = "prediction_model/ppv_npv_results.xlsx"
ppv_npv_df = pd.read_excel(ppv_npv_results_file_path)

## Simulation

### Parameters

In [5]:
seed = 42
slot_time = 20
num_serves = 1
work_hours = 10
overbooking = 1.0
simulation_days = 7
num_slots_byday = (60//slot_time) * work_hours
available_slots = num_slots_byday* simulation_days 

# extra percentage for appointments
extra_pct = 1.35
sample_size = int(np.ceil(available_slots + (available_slots * extra_pct)))

'''
# evaluando de 0.1 step hasta 0.9
thresholds_to_evaluate = []
for threshold1 in [x * 0.1 for x in range(1, 10)]:
    for threshold2 in [x * 0.1 for x in range(1, 10)]:
        thresholds_to_evaluate.append((round(threshold1,2), round(threshold2,2)))
'''

'''
# evaluando de 0.05 step hasta 0.5
thresholds_to_evaluate = []
for threshold1 in [x * 0.05 for x in range(2, 11)]:
    for threshold2 in [x * 0.05 for x in range(2, 11)]:
        thresholds_to_evaluate.append((round(threshold1,2), round(threshold2,2)))
'''

# evaluando de 0.05 step hasta 0.4
thresholds_to_evaluate = []
for threshold1 in [x * 0.05 for x in range(1, 9)]:
    for threshold2 in [x * 0.05 for x in range(1, 9)]:
        thresholds_to_evaluate.append((round(threshold1,2), round(threshold2,2)))

In [6]:
thresholds_to_evaluate  = [(0.05, 0.05), (0.1, 0.1), (0.15, 0.15), (0.2, 0.2), (0.25, 0.25), (0.3, 0.3)]

In [7]:
thresholds_to_evaluate  = [(0.1, 0.1), (0.15, 0.15), (0.2, 0.2), (0.25, 0.25)]

In [8]:
thresholds_to_evaluate 

[(0.1, 0.1), (0.15, 0.15), (0.2, 0.2), (0.25, 0.25)]

In [9]:
rule_name = "fountain"
param_overbooking = 1
sample_protected_pct = 0.50

In [10]:
# plot_prefix = f"{rule_name}{param_overbooking}_3high_sample"
# plot_prefix = f"{rule_name}{param_overbooking}_3high_{int(round(sample_protected_pct*100, 0))}_protected"
plot_prefix = f"{rule_name}{param_overbooking}_10high_{int(round(sample_protected_pct*100, 0))}protected"
num_replicas, iterations_per_schedule = 30, 30
verbose, debug_verbose = False, False

In [11]:
protected_true = [p for p in patients if p.protected]
protected_false = [p for p in patients if not p.protected]

print(f"Total patients: {len(patients)}")
print(f"Protected pool size: {len(protected_true)}")
print(f"Non-protected pool size: {len(protected_false)}")
print(f"Sample size total: {sample_size}")
print(f"Protected needed: {int(sample_size * 0.5)}")
print(f"Non-protected needed: {sample_size - int(sample_size * 0.5)}")
print(f"Protected pool sufficient: {len(protected_true) >= int(sample_size * 0.5)}")

Total patients: 104176
Protected pool size: 11291
Non-protected pool size: 92885
Sample size total: 494
Protected needed: 247
Non-protected needed: 247
Protected pool sufficient: True


In [12]:
prot_probas = [p.proba for p in patients if p.protected]
nonprot_probas = [p.proba for p in patients if not p.protected]

print(f"Protected proba: mean={np.mean(prot_probas):.4f}, std={np.std(prot_probas):.4f}")
print(f"NonProt proba:   mean={np.mean(nonprot_probas):.4f}, std={np.std(nonprot_probas):.4f}")

# Distribution comparison
from scipy.stats import ks_2samp
stat, pval = ks_2samp(prot_probas, nonprot_probas)
print(f"KS test: statistic={stat:.4f}, p-value={pval:.4g}")

Protected proba: mean=0.1128, std=0.0655
NonProt proba:   mean=0.1393, std=0.1579
KS test: statistic=0.1071, p-value=4.345e-101


In [ ]:
for threshold_iter, thresholds in enumerate(thresholds_to_evaluate):

    clear_output(wait=True)
    
    threshold_protected = thresholds[0]
    threshold_no_protected = thresholds[1]

    protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]['protected_ppv'].values[0]
    non_protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]['unprotected_ppv'].values[0]
    protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]['protected_npv'].values[0]
    non_protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]['unprotected_npv'].values[0]
    
    directory = f"simulation_outputs/automation_plots/{plot_prefix}_{num_replicas}_iter_plots/"
    if not os.path.exists(directory):
        os.makedirs(directory)

    directory = f"simulation_outputs/automation_plots/{plot_prefix}_{num_replicas}_iter_plots/{threshold_protected}_NPT_{threshold_no_protected}/"
    if not os.path.exists(directory):
        os.makedirs(directory)

    with open('patients_with_proba.pkl', 'rb') as f:
        patients_og = pickle.load(f) 
        
    all_measures = []
    convergence_values = []

    print(f"Simulating PT {threshold_protected} & NPT {threshold_no_protected} | [P PPV {protected_ppv:.4f}] [NP PPV {non_protected_ppv:.4f}] [P NPV {protected_npv:.4f}] [NP NPV {non_protected_npv:.4f}]")

    # region montecarlo
    convergence = False
    iterations = 0
    while not convergence and iterations < num_replicas:
        start = time.time()
        iterations += 1

        patients = copy.deepcopy(patients_og)

        if debug_verbose:
            print(f"After deepcopy {iterations} iterations | {time.time():.4f}")

        created_appointments = []
        created_appointments = create_appointments(num_serves, simulation_days, work_hours, slot_time)

        # ── Sample + composition diagnostic ──────────────────────────────────
        sampled_random_patient_list = random_patient_sample(patients, sample_size, sample_protected_pct, simulation_days)

        if iterations == 1:
            prot_in_sample    = [p for p in sampled_random_patient_list if p.protected]
            nonprot_in_sample = [p for p in sampled_random_patient_list if not p.protected]
            expected_high_proba_guaranteed = int(sample_size * sample_protected_pct) // 10
            high_proba_prot_count = sum(1 for p in prot_in_sample if p.proba > 0.3)
            print(f"  [sample] Protected  n={len(prot_in_sample)}, "
                  f"mean proba={np.mean([p.proba for p in prot_in_sample]):.4f}, "
                  f"above th({threshold_protected})={sum(1 for p in prot_in_sample if p.proba > threshold_protected)}")
            print(f"  [sample] NonProt    n={len(nonprot_in_sample)}, "
                  f"mean proba={np.mean([p.proba for p in nonprot_in_sample]):.4f}, "
                  f"above th({threshold_no_protected})={sum(1 for p in nonprot_in_sample if p.proba > threshold_no_protected)}")
            print(f"  [sample] High-proba protected (>0.3): {high_proba_prot_count} "
                  f"(guaranteed minimum: {expected_high_proba_guaranteed})")

        if debug_verbose:
            print(f"IDs in random_patient_list: {[patient.id for patient in sampled_random_patient_list]}")

        # ── Rule assignment + pairing diagnostic ─────────────────────────────
        appointments_from_rule, num_refused = call_a_rule(
            sampled_random_patient_list, created_appointments,
            rule_name, model,
            threshold_protected, threshold_no_protected,
            overbooking_level=param_overbooking
        )

        paired_prot_prot       = 0
        paired_nonprot_nonprot = 0
        paired_mixed           = 0
        total_double_slots     = 0
        patient_map = {p.id: p for p in sampled_random_patient_list}

        for server in appointments_from_rule:
            for day in server:
                for slot in day:
                    if isinstance(slot, list):
                        occupants = [pid for pid in slot if pid is not None]
                        if len(occupants) == 2:
                            total_double_slots += 1
                            p1 = patient_map[occupants[0]]
                            p2 = patient_map[occupants[1]]
                            if p1.protected and p2.protected:
                                paired_prot_prot += 1
                            elif (not p1.protected) and (not p2.protected):
                                paired_nonprot_nonprot += 1
                            else:
                                paired_mixed += 1

        if iterations == 1 and threshold_iter == 0:
            print(f"  [pairing] total double-booked slots : {total_double_slots}")
            print(f"  [pairing] Prot-Prot                : {paired_prot_prot}")
            print(f"  [pairing] NonProt-NonProt           : {paired_nonprot_nonprot}")
            print(f"  [pairing] Mixed                     : {paired_mixed}")

        if debug_verbose:
            print(f"IDs RandomPatientList after calling rule: {[patient.id for patient in sampled_random_patient_list]}")
            print(f"\nNumber of patients not assigned {num_refused}")

        # ── Inner simulation loop ─────────────────────────────────────────────
        for iter in range(iterations_per_schedule):

            appointments = copy.deepcopy(appointments_from_rule) 
            random_patient_list = copy.deepcopy(sampled_random_patient_list)

            attendance_random_patient_list = establish_attendance(
                random_patient_list,
                protected_ppv, non_protected_ppv,
                protected_npv, non_protected_npv,
                threshold_protected, threshold_no_protected
            )

            clinic = Clinic(attendance_random_patient_list, appointments, slot_time)
            clinic.simulation()
            measures = clinic.get_measures()

            if debug_verbose:
                print(f"Appointments after scheduling simulation: {appointments}")
                for server in appointments:
                    for i, day in enumerate(server):
                        print(f"\nD{i}")
                        for j, slot in enumerate(day):
                            for appoint_test in slot:
                                print(f"P{appoint_test}|ID{random_patient_list[appoint_test].id}|{random_patient_list[appoint_test].num_slot}", end=", ")
                print("measures:", measures)

            all_measures.append(measures)

            del appointments
            del random_patient_list
            del attendance_random_patient_list
            del clinic

        # -- Convergence check on per-replica aggregates ---------------------------
        # `all_measures` contains `iterations * iterations_per_schedule` rows, where
        # the inner `iterations_per_schedule` rows within each replica share the same
        # schedule (only attendance realization differs). Those rows are NOT
        # independent. For an honest convergence criterion, aggregate to one record
        # per replica, then check margin/mean on the aggregated series.
        if iterations >= 15:
            _tmp_df = pd.DataFrame(all_measures).copy()
            _tmp_df["replica_id"] = np.repeat(
                np.arange(iterations), iterations_per_schedule
            )[:len(_tmp_df)]

            # Aggregate inner-iteration records into one per replica (mean of the
            # attendance-realization distribution for that schedule).
            _replica_records = (
                _tmp_df
                .drop(columns=["idle_time_server", "over_time"], errors="ignore")
                .groupby("replica_id")
                .mean(numeric_only=True)
                .to_dict(orient="records")
            )

            convergence, diff = check_convergence_mean(
                _replica_records,
                converge_mean=0.02,   # tight: need to resolve ~2% effect sizes
                verbose=verbose,
            )
            convergence_values.append(diff)

            if verbose:
                print(f"  [convergence] iter {iterations}: worst margin/mean = "
                    f"{diff:.4f} ({'CONVERGED' if convergence else 'continuing'})")

        end = time.time()
        print(f"Ran Successfully Replica {iterations}/{num_replicas} in {end-start:.2f}s")

        del created_appointments
        del sampled_random_patient_list
        del patients

    print(f"* * * Ran Successfully All Replicas!!!\n")
    # endregion

    # region metric_calculations
    measures_df = pd.DataFrame(all_measures)
    summary_measures = get_margin_errors(all_measures)
    summary_measures_df = calculate_summary(measures_df)

    idle_time_mean        = summary_measures_df['mean'][0] / 7
    idle_time_lower_bound = summary_measures_df['confidence_interval'][0][0] / 7
    idle_time_upper_bound = summary_measures_df['confidence_interval'][0][1] / 7

    over_time_mean        = summary_measures_df['mean'][1] / 7
    over_time_lower_bound = summary_measures_df['confidence_interval'][1][0] / 7
    over_time_upper_bound = summary_measures_df['confidence_interval'][1][1] / 7

    no_shows_mean         = summary_measures_df['mean'][2] / 7
    no_shows_lower_bound  = summary_measures_df['confidence_interval'][2][0] / 7
    no_shows_upper_bound  = summary_measures_df['confidence_interval'][2][1] / 7

    patient_wt_row         = summary_measures_df[summary_measures_df["column"] == "patient_waiting_time"].iloc[0]
    patient_wt_mean        = patient_wt_row["mean"]
    patient_wt_lower_bound = patient_wt_row["confidence_interval"][0]
    patient_wt_upper_bound = patient_wt_row["confidence_interval"][1]

    # -- Per-replica aggregation for hypothesis-test series --------------------
    # Each replica runs one schedule and `iterations_per_schedule` attendance
    # realizations on that schedule. The realizations share the same rule
    # decisions and are therefore correlated. For valid statistical inference,
    # aggregate to the replica level first, then treat replicas as the
    # independent sample.
    measures_df["replica_id"] = np.repeat(
        np.arange(iterations), iterations_per_schedule
    )[:len(measures_df)]

    replica_grouped = (
        measures_df
        .groupby("replica_id")
        .agg(
            prot_wt_total    = ("protected_overbooked_waiting_time",     "sum"),
            nonprot_wt_total = ("non_protected_overbooked_waiting_time", "sum"),
            prot_ob_n        = ("protected_overbooked_patients",         "sum"),
            nonprot_ob_n     = ("non_protected_overbooked_patients",     "sum"),
            prot_attend      = ("protected_assistance",                  "sum"),
            nonprot_attend   = ("non_protected_assistance",              "sum"),
            conflict_prot    = ("conflict_slots_protected",              "sum"),
            conflict_nonprot = ("conflict_slots_non_protected",          "sum"),
        )
        .reset_index()
    )

    # Per-replica mean overbooked waiting time (stable denominator across the
    # replica's 30 inner realizations). Drop replicas with zero overbooked
    # patients — they carry no information for this metric.
    ob_prot_wt = (
        replica_grouped["prot_wt_total"] /
        replica_grouped["prot_ob_n"].replace(0, np.nan)
    ).dropna()
    ob_non_prot_wt = (
        replica_grouped["nonprot_wt_total"] /
        replica_grouped["nonprot_ob_n"].replace(0, np.nan)
    ).dropna()

    # Per-replica conflict rate (cleaner primary metric — unbounded by slot_time
    # and directly measures exposure to conflict events).
    conflict_rate_prot_per_replica = (
        replica_grouped["conflict_prot"] /
        replica_grouped["prot_attend"].replace(0, np.nan)
    ).dropna()
    conflict_rate_nonprot_per_replica = (
        replica_grouped["conflict_nonprot"] /
        replica_grouped["nonprot_attend"].replace(0, np.nan)
    ).dropna()

    protected_wt_mean,     protected_wt_margin     = confidence_interval(ob_prot_wt)
    non_protected_wt_mean, non_protected_wt_margin = confidence_interval(ob_non_prot_wt)

    total_attended_mean,  total_attended_margin  = confidence_interval(measures_df["total_attended_patients"])

    conflict_prot_mean,     conflict_prot_margin     = confidence_interval(measures_df["conflict_slots_protected"])
    conflict_nonprot_mean,  conflict_nonprot_margin  = confidence_interval(measures_df["conflict_slots_non_protected"])

    conflict_rate_prot_mean,    conflict_rate_prot_margin    = confidence_interval(measures_df["conflict_rate_protected"])
    conflict_rate_nonprot_mean, conflict_rate_nonprot_margin = confidence_interval(measures_df["conflict_rate_non_protected"])

    one_sided_correction_df     = ttest(ob_prot_wt, ob_non_prot_wt, correction=True,  alternative='greater',   confidence=0.95)
    one_sided_non_correction_df = ttest(ob_prot_wt, ob_non_prot_wt, correction=False, alternative='greater',   confidence=0.95)
    two_sided_correction_df     = ttest(ob_prot_wt, ob_non_prot_wt, correction=True,  alternative='two-sided', confidence=0.95)
    two_sided_non_correction_df = ttest(ob_prot_wt, ob_non_prot_wt, correction=False, alternative='two-sided', confidence=0.95)

    one_sided_correction_p_val     = one_sided_correction_df["p_val"].values[0]
    one_sided_correction_t         = one_sided_correction_df["T"].values[0]
    one_sided_non_correction_p_val = one_sided_non_correction_df["p_val"].values[0]
    one_sided_non_correction_t     = one_sided_non_correction_df["T"].values[0]
    two_sided_correction_p_val     = two_sided_correction_df["p_val"].values[0]
    two_sided_correction_t         = two_sided_correction_df["T"].values[0]
    two_sided_non_correction_p_val = two_sided_non_correction_df["p_val"].values[0]
    two_sided_non_correction_t     = two_sided_non_correction_df["T"].values[0]

    # -- Conflict-rate t-tests (primary mechanistic metric) --------------------
    cr_one_sided_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=True, alternative='greater', confidence=0.95
    )
    cr_two_sided_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=True, alternative='two-sided', confidence=0.95
    )
    cr_one_sided_non_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=False, alternative='greater', confidence=0.95
    )
    cr_two_sided_non_correction_df = ttest(
        conflict_rate_prot_per_replica, conflict_rate_nonprot_per_replica,
        correction=False, alternative='two-sided', confidence=0.95
    )

    cr_one_sided_correction_p_val     = cr_one_sided_correction_df["p_val"].values[0]
    cr_one_sided_correction_t         = cr_one_sided_correction_df["T"].values[0]
    cr_two_sided_correction_p_val     = cr_two_sided_correction_df["p_val"].values[0]
    cr_two_sided_correction_t         = cr_two_sided_correction_df["T"].values[0]
    cr_one_sided_non_correction_p_val = cr_one_sided_non_correction_df["p_val"].values[0]
    cr_one_sided_non_correction_t     = cr_one_sided_non_correction_df["T"].values[0]
    cr_two_sided_non_correction_p_val = cr_two_sided_non_correction_df["p_val"].values[0]
    cr_two_sided_non_correction_t     = cr_two_sided_non_correction_df["T"].values[0]

    # region summary_df_and_excel
    summary_row = {
        # ══════════════════════════════════════════════════════════════════════
        # 1. EXPERIMENT IDENTITY — what scenario is this row?
        # ══════════════════════════════════════════════════════════════════════
        "rule":                              f"{rule_name.replace('_', ' ').title()} {param_overbooking}",
        "protected_proportion":              sample_protected_pct,
        "unprotected_proportion":            1 - sample_protected_pct,
        "protected_threshold":               threshold_protected,
        "unprotected_threshold":             threshold_no_protected,
        "iterations":                        f"{iterations} replicas | {iterations_per_schedule} iter",

        # ══════════════════════════════════════════════════════════════════════
        # 2. BIAS SOURCE — the model-calibration asymmetry being tested
        # ══════════════════════════════════════════════════════════════════════
        "protected_ppv":                     protected_ppv,
        "unprotected_ppv":                   non_protected_ppv,
        "protected_npv":                     protected_npv,
        "unprotected_npv":                   non_protected_npv,

        # ══════════════════════════════════════════════════════════════════════
        # 3. PRIMARY OUTCOME — conflict rate (cleanest mechanistic signal)
        # ══════════════════════════════════════════════════════════════════════
        "conflict_rate_protected_mean":      conflict_rate_prot_mean,
        "conflict_rate_non_protected_mean":  conflict_rate_nonprot_mean,
        "conflict_rate_protected_lower":     conflict_rate_prot_mean    - conflict_rate_prot_margin,
        "conflict_rate_non_protected_lower": conflict_rate_nonprot_mean - conflict_rate_nonprot_margin,
        "conflict_rate_protected_upper":     conflict_rate_prot_mean    + conflict_rate_prot_margin,
        "conflict_rate_non_protected_upper": conflict_rate_nonprot_mean + conflict_rate_nonprot_margin,

        # Conflict-rate hypothesis tests (primary)
        "cr_one_sided_pval_corr":            cr_one_sided_correction_p_val,
        "cr_two_sided_pval_corr":            cr_two_sided_correction_p_val,
        "cr_one_sided_t_corr":               cr_one_sided_correction_t,
        "cr_two_sided_t_corr":               cr_two_sided_correction_t,
        "cr_one_sided_pval_non_corr":        cr_one_sided_non_correction_p_val,
        "cr_two_sided_pval_non_corr":        cr_two_sided_non_correction_p_val,
        "cr_one_sided_t_non_corr":           cr_one_sided_non_correction_t,
        "cr_two_sided_t_non_corr":           cr_two_sided_non_correction_t,

        # ══════════════════════════════════════════════════════════════════════
        # 4. SECONDARY OUTCOME — overbooked waiting time (bounded by slot_time)
        # ══════════════════════════════════════════════════════════════════════
        "ob_protected_wt_mean":              protected_wt_mean,
        "ob_non_protected_wt_mean":          non_protected_wt_mean,
        "ob_protected_wt_lower":             protected_wt_mean     - protected_wt_margin,
        "ob_non_protected_wt_lower":         non_protected_wt_mean - non_protected_wt_margin,
        "ob_protected_wt_upper":             protected_wt_mean     + protected_wt_margin,
        "ob_non_protected_wt_upper":         non_protected_wt_mean + non_protected_wt_margin,

        # Overbooked-WT hypothesis tests (secondary)
        "one_sided_pval_corr":               one_sided_correction_p_val,
        "two_sided_pval_corr":               two_sided_correction_p_val,
        "one_sided_t_corr":                  one_sided_correction_t,
        "two_sided_t_corr":                  two_sided_correction_t,
        "one_sided_pval_non_corr":           one_sided_non_correction_p_val,
        "two_sided_pval_non_corr":           two_sided_non_correction_p_val,
        "one_sided_t_non_corr":              one_sided_non_correction_t,
        "two_sided_t_non_corr":              two_sided_non_correction_t,

        # ══════════════════════════════════════════════════════════════════════
        # 5. MECHANISM DIAGNOSTICS — how the effect is produced
        # ══════════════════════════════════════════════════════════════════════
        # Conflict-slot event counts (raw numerator of conflict rate)
        "conflict_slots_protected_mean":      conflict_prot_mean,
        "conflict_slots_non_protected_mean":  conflict_nonprot_mean,
        "conflict_slots_protected_lower":     conflict_prot_mean    - conflict_prot_margin,
        "conflict_slots_non_protected_lower": conflict_nonprot_mean - conflict_nonprot_margin,
        "conflict_slots_protected_upper":     conflict_prot_mean    + conflict_prot_margin,
        "conflict_slots_non_protected_upper": conflict_nonprot_mean + conflict_nonprot_margin,

        # Pairing composition — whose slots conflict with whose?
        "total_double_slots":                total_double_slots,
        "paired_prot_prot":                  paired_prot_prot,
        "paired_nonprot_nonprot":            paired_nonprot_nonprot,
        "paired_mixed":                      paired_mixed,

        # ══════════════════════════════════════════════════════════════════════
        # 6. OPERATIONAL CONTEXT — clinic-level realism sanity checks
        # ══════════════════════════════════════════════════════════════════════
        "overall_patient_wt_mean":           patient_wt_mean,
        "overall_patient_wt_lower":          patient_wt_lower_bound,
        "overall_patient_wt_upper":          patient_wt_upper_bound,
        "total_attended_mean":               total_attended_mean,
        "total_attended_lower":              total_attended_mean - total_attended_margin,
        "total_attended_upper":              total_attended_mean + total_attended_margin,
        "idle_time_mean":                    idle_time_mean,
        "idle_time_lower":                   idle_time_lower_bound,
        "idle_time_upper":                   idle_time_upper_bound,
        "over_time_mean":                    over_time_mean,
        "over_time_lower":                   over_time_lower_bound,
        "over_time_upper":                   over_time_upper_bound,
    }

    if threshold_iter == 0:
        results_summary_rows = []
    results_summary_rows.append(summary_row)

    results_summary_df = pd.DataFrame(results_summary_rows)

    base_dir = Path("simulation_outputs/excel_files/post_thesis_iterations")
    base_dir.mkdir(parents=True, exist_ok=True)
    summary_path = base_dir / f"{plot_prefix}_{num_replicas}_replicas_{iterations_per_schedule}_iter_summary.xlsx"

    with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
        results_summary_df.to_excel(writer, index=False, sheet_name="Summary")

        worksheet = writer.sheets["Summary"]
        for col_cells in worksheet.columns:
            max_len = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in col_cells
            )
            worksheet.column_dimensions[col_cells[0].column_letter].width = max(max_len + 2, 12)

        from openpyxl.styles import PatternFill, Font, Alignment
        header_fill = PatternFill(start_color="00024A", end_color="00024A", fill_type="solid")
        header_font = Font(color="E5F2FC", bold=True)
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")

        green_fill  = PatternFill(start_color="AEFB7F", end_color="AEFB7F", fill_type="solid")
        yellow_fill = PatternFill(start_color="FFFD8D", end_color="FFFD8D", fill_type="solid")
        red_fill    = PatternFill(start_color="FA8072", end_color="FA8072", fill_type="solid")
        blue_fill   = PatternFill(start_color="CDEEFD", end_color="CDEEFD", fill_type="solid")

        pval_col_idx = results_summary_df.columns.get_loc("two_sided_pval_corr") + 1
        pt_col_idx   = results_summary_df.columns.get_loc("protected_threshold") + 1
        npt_col_idx  = results_summary_df.columns.get_loc("unprotected_threshold") + 1

        for row_idx, row in enumerate(results_summary_rows, start=2):
            pval = row["two_sided_pval_corr"]
            pt   = row["protected_threshold"]
            npt  = row["unprotected_threshold"]
            worksheet.cell(row=row_idx, column=pval_col_idx).fill = (
                blue_fill if pval > 0.05 else PatternFill()
            )
            if pval > 0.05:
                threshold_fill = green_fill if pt > npt else (red_fill if pt < npt else yellow_fill)
                worksheet.cell(row=row_idx, column=pt_col_idx).fill  = threshold_fill
                worksheet.cell(row=row_idx, column=npt_col_idx).fill = threshold_fill

    print(f"  [summary] Saved {len(results_summary_rows)} threshold pair(s) → {summary_path}")
    # endregion

Simulating PT 0.25 & NPT 0.25 | [P PPV 0.2716] [NP PPV 0.5525] [P NPV 0.8935] [NP NPV 0.9109]
  [sample] Protected  n=247, mean proba=0.1410, above th(0.25)=34
  [sample] NonProt    n=247, mean proba=0.1359, above th(0.25)=25
  [sample] High-proba protected (>0.3): 28 (guaranteed minimum: 24)
Ran Successfully Replica 1/30 in 5.28s
Ran Successfully Replica 2/30 in 5.10s
Ran Successfully Replica 3/30 in 4.91s
Ran Successfully Replica 4/30 in 4.96s
Ran Successfully Replica 5/30 in 5.24s
Ran Successfully Replica 6/30 in 5.00s
Ran Successfully Replica 7/30 in 5.08s
Ran Successfully Replica 8/30 in 5.08s
Ran Successfully Replica 9/30 in 4.95s
Ran Successfully Replica 10/30 in 5.11s
Ran Successfully Replica 11/30 in 4.88s
Ran Successfully Replica 12/30 in 4.79s
Ran Successfully Replica 13/30 in 4.91s
Ran Successfully Replica 14/30 in 4.84s
Ran Successfully Replica 15/30 in 4.92s
Ran Successfully Replica 16/30 in 4.70s
Ran Successfully Replica 17/30 in 4.85s
Ran Successfully Replica 18/30 in 4

### Process

In [ ]:
print(f"n_high_protected needed: {max(1, int(sample_size * 0.5) // 10)}")
print(f"high_proba_protected pool size: {len([p for p in patients if p.protected and p.proba > 0.5])}")